# Neo4j Index Setup for legal-legislation-explorer

This notebook creates the recommended indexes for the query patterns in your app, using idempotent `IF NOT EXISTS` statements.

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

assert NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD, 'Missing Neo4j env vars'
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print('Connected to', NEO4J_URI, 'database=', NEO4J_DATABASE)

Connected to neo4j+s://d4b862fa.databases.neo4j.io database= neo4j


In [2]:
# Inspect current indexes
with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run("SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties, state RETURN * ORDER BY name").data()

print(f'Existing indexes: {len(rows)}')
rows[:20]

Existing indexes: 17


[{'name': 'chap_id_unique',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chapter'],
  'properties': ['id'],
  'state': 'ONLINE'},
 {'name': 'cit_id_unique',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Citation'],
  'properties': ['id'],
  'state': 'ONLINE'},
 {'name': 'com_id_unique',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Commentary'],
  'properties': ['id'],
  'state': 'ONLINE'},
 {'name': 'en_id_unique',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ExplanatoryNotes'],
  'properties': ['id'],
  'state': 'ONLINE'},
 {'name': 'ep_id_unique',
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['ExplanatoryNotesParagraph'],
  'properties': ['id'],
  'state': 'ONLINE'},
 {'name': 'index_1b9dcc97',
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'state': 'ONLINE'},
 {'name': 'index_460996c0',
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labels

## Recommended indexes

These target the app's hottest filters: `Legislation.uri` (`CONTAINS` + exact), `Part.order`, and temporal boundaries on `Paragraph`.

Optional indexes are included for generic URI lookups across other labels and fuzzy title/URI search.

In [3]:
index_statements = [
    # High-impact
    "CREATE TEXT INDEX legislation_uri_text IF NOT EXISTS FOR (l:Legislation) ON (l.uri)",
    "CREATE RANGE INDEX legislation_uri_range IF NOT EXISTS FOR (l:Legislation) ON (l.uri)",
    "CREATE RANGE INDEX part_order_range IF NOT EXISTS FOR (p:Part) ON (p.order)",
    "CREATE RANGE INDEX paragraph_restrict_start_date_range IF NOT EXISTS FOR (p:Paragraph) ON (p.restrict_start_date)",
    "CREATE RANGE INDEX paragraph_restrict_end_date_range IF NOT EXISTS FOR (p:Paragraph) ON (p.restrict_end_date)",

    "CREATE TEXT INDEX paragraph_uri_text IF NOT EXISTS FOR (p:Paragraph) ON (p.uri)",
    "CREATE TEXT INDEX section_uri_text IF NOT EXISTS FOR (s:Section) ON (s.uri)",
    "CREATE FULLTEXT INDEX legislation_title_uri_fulltext IF NOT EXISTS FOR (l:Legislation) ON EACH [l.title, l.uri]",
]

with driver.session(database=NEO4J_DATABASE) as session:
    for stmt in index_statements:
        session.run(stmt).consume()
        print('OK:', stmt)

OK: CREATE TEXT INDEX legislation_uri_text IF NOT EXISTS FOR (l:Legislation) ON (l.uri)
OK: CREATE RANGE INDEX legislation_uri_range IF NOT EXISTS FOR (l:Legislation) ON (l.uri)
OK: CREATE RANGE INDEX part_order_range IF NOT EXISTS FOR (p:Part) ON (p.order)
OK: CREATE RANGE INDEX paragraph_restrict_start_date_range IF NOT EXISTS FOR (p:Paragraph) ON (p.restrict_start_date)
OK: CREATE RANGE INDEX paragraph_restrict_end_date_range IF NOT EXISTS FOR (p:Paragraph) ON (p.restrict_end_date)
OK: CREATE TEXT INDEX paragraph_uri_text IF NOT EXISTS FOR (p:Paragraph) ON (p.uri)
OK: CREATE TEXT INDEX section_uri_text IF NOT EXISTS FOR (s:Section) ON (s.uri)
OK: CREATE FULLTEXT INDEX legislation_title_uri_fulltext IF NOT EXISTS FOR (l:Legislation) ON EACH [l.title, l.uri]


In [4]:
# Verify index state
created_names = [
    'legislation_uri_text',
    'legislation_uri_range',
    'part_order_range',
    'paragraph_restrict_start_date_range',
    'paragraph_restrict_end_date_range',
    'paragraph_uri_text',
    'section_uri_text',
    'legislation_title_uri_fulltext',
]

with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(
        "SHOW INDEXES YIELD name, type, state, populationPercent, labelsOrTypes, properties WHERE name IN $names RETURN * ORDER BY name",
        names=created_names,
    ).data()

rows

[{'name': 'legislation_title_uri_fulltext',
  'type': 'FULLTEXT',
  'state': 'POPULATING',
  'populationPercent': 0.0,
  'labelsOrTypes': ['Legislation'],
  'properties': ['title', 'uri']},
 {'name': 'legislation_uri_text',
  'type': 'TEXT',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'labelsOrTypes': ['Legislation'],
  'properties': ['uri']},
 {'name': 'paragraph_restrict_end_date_range',
  'type': 'RANGE',
  'state': 'POPULATING',
  'populationPercent': 4.199999809265137,
  'labelsOrTypes': ['Paragraph'],
  'properties': ['restrict_end_date']},
 {'name': 'paragraph_restrict_start_date_range',
  'type': 'RANGE',
  'state': 'POPULATING',
  'populationPercent': 4.199999809265137,
  'labelsOrTypes': ['Paragraph'],
  'properties': ['restrict_start_date']},
 {'name': 'paragraph_uri_text',
  'type': 'TEXT',
  'state': 'POPULATING',
  'populationPercent': 0.0,
  'labelsOrTypes': ['Paragraph'],
  'properties': ['uri']},
 {'name': 'part_order_range',
  'type': 'RANGE',
  'state': 'ONL

## Notes

- `TEXT` indexes are used for string search patterns like `CONTAINS`.
- `RANGE` indexes are used for equality/range comparisons and date boundaries.
- If you later switch title/URI fuzzy search to `db.index.fulltext.queryNodes`, the fulltext index will provide stronger performance for those queries.

In [5]:
driver.close()
print('Done.')

Done.
